# 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
## Custom functions
import FockSystem.FockSystem as fst
import Analysis.transport_tools as tu
from Analysis.systems import kitaev_chain, kramers_chain
from itertools import product,combinations,permutations
import cvxpy as cp
from IPython.display import display, Math,Markdown
from copy import deepcopy
from itertools import chain

In [ ]:
import random

In [ ]:
from FockSystem.c import fermion_operations as fo

In [ ]:
def print_md(item):
    display(Markdown(item._repr_markdown_()))

In [ ]:
def commute(operator_1,operator_2):
    commutator = operator_1*operator_2 - operator_2*operator_1
    commutator.remove_zero_weight()
    return commutator

In [ ]:
def act_oper_on_state(Opers, State):
    states = State.states
    filt_zero = np.where(State.weights != 0)
    relevant_states = np.array(states[filt_zero])
    relevant_phi = State.weights[filt_zero]
    
    result_state = np.zeros(len(states),dtype=complex)

    for operators,oper_weight in zip(Opers.oper_list,Opers.weights):
      
        new_states = relevant_states
        new_signs = np.full(len(relevant_states),1)
        if isinstance(operators,list):
            for op in operators:
                new_states,sign = fs.act_oper(op, new_states)
                new_signs *= sign

        weights = np.round(relevant_phi*new_signs*oper_weight,14)

        row_idx=0
        for state,weight in zip(new_states,weights):
            if state>-1 and (state in State.states):
                result_state[State.hashed[state]] += weight
    return fst.FockStates(states = states, weights = result_state)

In [ ]:
def build_column_matrix(operator_options, phi_A, basis):
    states = basis.states
    filt_zero = np.where(phi_A != 0)
    relevant_states = np.array(states[filt_zero])
    relevant_phi = phi_A[filt_zero]
    
    column_matrix = np.zeros((len(basis.states), len(operator_options)),dtype=complex)
    
    for col_idx,operators in enumerate(operator_options):
        new_states = relevant_states
        new_signs = np.full(len(relevant_states),1)
        for op in operators:
            new_states,sign = fs.act_oper(op, new_states)
            new_signs *= sign

        weights = relevant_phi*new_signs

        row_idx=0
        for state,weight in zip(new_states,weights):
            if state>-1 and (state in basis.states):
                column_matrix[basis.hashed[state]][col_idx] = weight

    return column_matrix

In [ ]:
def build_column_matrix_operwise(operator_options, phi_A, basis):
    states = basis.states
    filt_zero = np.where(phi_A != 0)
    relevant_states = np.array(states[filt_zero])
    relevant_phi = phi_A[filt_zero]

    state_as_oper = [fs.state_to_oper_list(i) for i in relevant_states]
    state_operator = fst.OperSequence(state_as_oper, weights=relevant_phi, bypass_parse=True)

    options_as_opers = [fst.OperSequence([operator_options[i]], bypass_parse=True) for i in range(len(operator_options))]

    column_matrix = np.zeros((len(operator_options), len(operator_options)),dtype=complex)
    for col_idx,basis_operator in enumerate(options_as_opers):
        product_state = basis_operator*state_operator
        for oper_list, weight in zip(product_state.oper_list, product_state.weights):
            if oper_list in operator_options:
                column_matrix[operator_options.index(oper_list)][col_idx] = weight
            else:
                if oper_list == 1:
                    oper_list = []
                    column_matrix[operator_options.index(oper_list)][col_idx] = weight
              
    return column_matrix

In [ ]:
def state_as_oper_vector(phi, basis,operator_options):
    states= basis.states
    filt_zero = np.where(phi != 0)
    relevant_states = np.array(states[filt_zero])
    relevant_phi = phi[filt_zero]

    states_as_opers = [fs.state_to_oper_list(i) for i in relevant_states]
    vector = np.zeros(len(operator_options),dtype=complex)
    for oper_list,weight in zip(states_as_opers, relevant_phi):
        if oper_list in operator_options:
            vector[operator_options.index(oper_list)] = weight
    return vector

In [ ]:
def is_hermitian(operator):
    conj = ~operator
    conj.normal_order()
    return conj == operator

In [ ]:
def constrained_operator_in_oper_space(phi_A, phi_B, operator_options,basis):
    ###########################
    ## GENERATE PHI MATRICES ##
    ###########################
    col_matrix_A = build_column_matrix_operwise(operator_options, phi_A, basis)
    col_matrix_B = build_column_matrix_operwise(operator_options, phi_B, basis)

    phi_A_as_vector = state_as_oper_vector(phi_A, basis,operator_options)
    phi_B_as_vector = state_as_oper_vector(phi_B, basis,operator_options)
    
    
    M_stacked = np.vstack([
        col_matrix_A,   # shape (D, M)
        col_matrix_B    # shape (D, M)
    ],dtype=complex)           # shape (2D, M)
    
    phi_stacked = np.concatenate([
        phi_B_as_vector,       # |B⟩
        phi_A_as_vector        # |A⟩
    ],dtype=complex)           # shape (2D,)

    ####################################
    ##   Solve constrained problem   ##
    ####################################
    A = cp.Variable(len(operator_options))
    objective = cp.Minimize(cp.sum_squares(M_stacked @ A - phi_stacked))
    
    constraints = extract_constraints(operator_options, A)
    
    problem = cp.Problem(objective, constraints)
    problem.solve()
    
    ###########################
    ##    Return Operator    ##
    ###########################
    
    result = np.round(A.value,14)

    filt_zeros = np.where(result != 0)[0]
    weights = result[filt_zeros]
    opers = [operator_options[i] for i in filt_zeros]
    
    operator = fst.OperSequence(opers, weights=list(weights), bypass_parse=True)
    return operator

In [ ]:
def map_to_new_basis(phi, restr_basis,new_basis):
    new_vec = np.zeros(len(new_basis.states),dtype=complex)
    for idx, p in enumerate(phi):
        new_idx = list(new_basis.states).index(restr_basis.states[idx])
        new_vec[new_idx] = p
    return np.round(new_vec,10)

In [ ]:
def get_combinations(lst, n):
    return list(combinations(lst, n))
    
def generate_combinations_restr(pos_ops):
    all_options = []
    for i in range(1,len(pos_ops)+1):
        combs = get_combinations(pos_ops, i)
        all_options.extend([fs.normal_order_naive(list(c))[0] for c in combs if len(c)%2 == 1 and len(c) < 7])
    return all_options

def generate_combinations(pos_ops):
    all_options = []
    for i in range(1,len(pos_ops)+1):
        combs = get_combinations(pos_ops, i)
        all_options.extend([fs.normal_order_naive(list(c))[0] for c in combs])
    return all_options

#options = gamma.oper_list

In [ ]:
def extract_constained_indices(operator_options):
    constrained_indices = []
    for idx_1,oper in enumerate(operator_options):
        conjugate_list = fs.conjugate_list(deepcopy(oper))
        weights,conj_list = fo.normal_order([1],[deepcopy(conjugate_list)])
        sign = weights[-1]
        if conj_list[0] in operator_options:
            idx_2 = operator_options.index(conj_list[0])
            if (idx_2,idx_1,sign) not in constrained_indices:
                constrained_indices.append((idx_1, idx_2, sign))
    return constrained_indices

def extract_constraints(operator_options, variable):
    constrained_indices = extract_constained_indices(operator_options)

    constraints = []
    for i, j, s in constrained_indices:
        if s == 1:
            constraints.append(variable[i] == cp.conj(variable[j]))
        elif s == -1:
            constraints.append(variable[i] == -cp.conj(variable[j]))

    return constraints

In [ ]:
def constrained_operator(phi_A, phi_B, operator_options,basis, commutator_oper = None):
    ###########################
    ## GENERATE PHI MATRICES ##
    ###########################
    col_matrix_A = build_column_matrix(operator_options, phi_A, basis)
    col_matrix_B = build_column_matrix(operator_options, phi_B, basis)
    
    M_stacked = np.round(np.vstack([
        col_matrix_A,   # shape (D, M)
        col_matrix_B    # shape (D, M)
    ],dtype=complex),10)           # shape (2D, M)
    
    phi_stacked = np.concatenate([
        phi_B,       # |B⟩
        phi_A        # |A⟩
    ],dtype=complex)           # shape (2D,)

    ####################################
    ##   Solve constrained problem   ##
    ####################################
    A = cp.Variable(len(operator_options))
    objective = cp.Minimize(cp.sum_squares(M_stacked @ A - phi_stacked))
    
    constraints = extract_constraints(operator_options, A)
    if commutator_oper is not None:
        commute_matrix = build_commutator_matrix(operator_options, commutator_oper)
        constraints.append(commute_matrix@A==0)
    
    problem = cp.Problem(objective, constraints)
    problem.solve(solver=cp.ECOS_BB)

    ###########################
    ##    Return Operator    ##
    ###########################
    result = np.round(A.value,8)
   

    filt_zeros = np.where(result != 0)[0]
    weights = result[filt_zeros]
    opers = [operator_options[i] for i in filt_zeros]
    
    operator = fst.OperSequence(opers, weights=list(weights), bypass_parse=True)
    return operator

In [ ]:
def min_constrained_operator(phi_A, phi_B, operator_options,basis, commutator_oper = None):
    ###########################
    ## GENERATE PHI MATRICES ##
    ###########################
    col_matrix_A = build_column_matrix(operator_options, phi_A, basis)
    col_matrix_B = build_column_matrix(operator_options, phi_B, basis)
    
    M_stacked = np.round(np.vstack([
        col_matrix_A,   # shape (D, M)
        col_matrix_B    # shape (D, M)
    ],dtype=complex),10)           # shape (2D, M)
    
    phi_stacked = np.concatenate([
        phi_B,       # |B⟩
        phi_A        # |A⟩
    ],dtype=complex)           # shape (2D,)

    ####################################
    ##   Solve constrained problem   ##
    ####################################
    A = cp.Variable(len(operator_options))
    objective = cp.Minimize(cp.norm1(A))
    
    constraints = extract_constraints(operator_options, A)
    constraints.append(M_stacked @ A - phi_stacked == 0)
    if commutator_oper is not None:
        commute_matrix = build_commutator_matrix(operator_options, commutator_oper)
        constraints.append(commute_matrix@A==0)
    
    problem = cp.Problem(objective, constraints)
    problem.solve(solver=cp.ECOS_BB)

    ###########################
    ##    Return Operator    ##
    ###########################
    if A.value is not None:
        result = np.round(A.value,8)
        
        filt_zeros = np.where(result != 0)[0]
        weights = result[filt_zeros]
        opers = [operator_options[i] for i in filt_zeros]
        
        operator = fst.OperSequence(opers, weights=list(weights), bypass_parse=True)
        return operator
    else:
        return False

In [ ]:
def test_operator(phi_A, phi_B, basis, operator, verbose=True):
    state_A = fst.FockStates(states = basis.states, weights = phi_A)
    state_B = fst.FockStates(states = basis.states, weights = phi_B)

    new_state_B = act_oper_on_state(operator, state_A)
    new_state_A = act_oper_on_state(operator, state_B)
    
    new_state_B.weights

    norm_A = np.linalg.norm(np.array(state_A.weights) - np.array(new_state_A.weights))
    norm_B = np.linalg.norm(np.array(state_B.weights) - np.array(new_state_B.weights))

    if verbose:
        display(Markdown(f'State_A: {state_A._repr_markdown_()}'))
        display(Markdown(f'State_B: {state_B._repr_markdown_()}'))
    
        display(Markdown(f"Acting on states with $\\gamma$ = {operator._repr_markdown_()}"))
        
        display(Markdown(f'$\\gamma |\\Phi_A$\u3009 '+ f'= {new_state_B._repr_markdown_()} (d = {np.round(norm_A,4)})'))
        display(Markdown(f'$\\gamma |\\Phi_B$\u3009 '+ f'= {new_state_A._repr_markdown_()} (d = {np.round(norm_B,4)})'))

        back_to_A = act_oper_on_state(operator**2, state_A)
        back_to_B = act_oper_on_state(operator**2, state_B)

        norm_A = np.linalg.norm(np.array(state_A.weights) - np.array(back_to_A.weights))
        norm_B = np.linalg.norm(np.array(state_B.weights) - np.array(back_to_B.weights))

        display(Markdown(f'$\\gamma^2 |\\Phi_A$\u3009 '+ f'= {back_to_A._repr_markdown_()} (d = {np.round(norm_A,4)})'))
        display(Markdown(f'$\\gamma^2 |\\Phi_B$\u3009 '+ f'= {back_to_B._repr_markdown_()} (d = {np.round(norm_B,4)})'))


    return np.round(norm_A,6),np.round(norm_B,6)

In [ ]:
empty_sequence = fst.OperSequence([],bypass_parse=True)
def iterate_to_optimal_operator(phi_A, phi_B, operator_options,basis,commutator_oper = None):
    start_operator = min_constrained_operator(phi_A,phi_B,operator_options,basis, commutator_oper=commutator_oper)
    conjugate_pairs = extract_constained_indices(start_operator.oper_list)
    
    useable_pairs = get_combinations(conjugate_pairs, len(conjugate_pairs)-1)
    optimal_list = []
    operators = []
    for possible_pairs in useable_pairs:
        reduced_oper_list = [start_operator.oper_list[i] for i in chain.from_iterable((t[0],t[1]) for t in possible_pairs)]
        new_operator = min_constrained_operator(phi_A,phi_B, reduced_oper_list, basis,commutator_oper=commutator_oper)
        if new_operator:
            norm_A, norm_B = test_operator(phi_A,phi_B, basis,new_operator,verbose=False)
            optimal_list.append(norm_A+norm_B)
            operators.append(new_operator)
        else:
            optimal_list.append(1000)
            operators.append(False)
        
    possible_paths = np.where(np.array(optimal_list) < 0.01)[0]  
    if len(possible_paths)>0:
        random_path_idx = random.choice(possible_paths)
        new_operator_options = [start_operator.oper_list[i] for i in chain.from_iterable((t[0],t[1]) for t in useable_pairs[random_path_idx])]
        return iterate_to_optimal_operator(phi_A,phi_B,new_operator_options,basis, commutator_oper = commutator_oper)

    else:
        return start_operator


##  Old solution from paper

In [ ]:
c_d_L = fst.OperSequence(0)
c_u_L = fst.OperSequence(2)
c_d_R = fst.OperSequence(4)
c_u_R = fst.OperSequence(6)
n_L_u = c_u_L*(~c_u_L)
n_L_d = c_d_L*(~c_d_L)
n_R_u = c_u_R*(~c_u_R)
n_R_d = c_d_R*(~c_d_R)
n_L = c_u_L*(~c_u_L) + c_d_L*(~c_d_L)

a_down_L = (-n_L_u+1)*(~c_d_L)
a_up_L = (-n_L_d+1)*(~c_u_L)
a_down_R = (-n_R_u+1)*(~c_d_R)
a_up_R = (-n_R_d+1)*(~c_u_R)

n_L_constr_up  = (~a_up_L)*(a_up_L)
n_L_constr = (~a_down_L)*(a_down_L) + (~a_up_L)*(a_up_L)

gamma_R_constr = -(-n_L_constr+1)*(a_up_R) - 1/np.sqrt(2)*(n_L_constr_up*(a_down_R) - (~a_up_L)*(a_down_L)*(a_up_R))
gamma_constr_paper = gamma_R_constr + (~gamma_R_constr)
gamma_R_up = -(-n_L+1)*(~c_u_R) - 1/np.sqrt(2)*(n_L_u*(~c_d_R) - c_u_L*(~c_d_L)*(~c_u_R))
gamma_from_paper = gamma_R_up + (~gamma_R_up)

In [ ]:
n_L

In [ ]:
c_up = fst.OperSequence(0)

In [ ]:
c_down = fst.OperSequence(2)

In [ ]:
maj = c_up + ~c_up

In [ ]:
t_2 = (c_up>>1)*((~c_up)>>2) + (c_down>>1)*((~c_down)>>2)

In [ ]:
t_2

In [ ]:
commute(gamma_from_paper,t_2)

In [ ]:
t_2

In [ ]:
comm = gamma_from_paper*n_L - n_L*gamma_from_paper
comm.remove_zero_weight()
comm

In [ ]:
gamma_from_paper

In [ ]:
gamma_constr**2

In [ ]:
gamma_constr_paper**2

In [ ]:
gamma_from_paper**2

## Two sites

In [ ]:
def build_commutator_matrix(operator_options,commutator_state):
    options_as_opers = [fst.OperSequence([operator_options[i]], bypass_parse=True) for i in range(len(operator_options))]
    column_matrix = np.zeros((len(operator_options), len(operator_options)),dtype=complex)
    
    for col_idx,basis_operator in enumerate(options_as_opers):
        commutator = basis_operator*commutator_state - commutator_state*basis_operator
        for oper_list, weight in zip(commutator.oper_list, commutator.weights):
            if oper_list in operator_options:
                column_matrix[operator_options.index(oper_list)][col_idx] = weight
            else:
                if oper_list == 1:
                    oper_list = []
                    column_matrix[operator_options.index(oper_list)][col_idx] = weight
              
    return column_matrix

In [ ]:
fs = fst.FockSystemBase()
N=2
pos_ops = np.arange(4*N)
options = generate_combinations_restr(pos_ops)
#options = gamma.oper_list

In [ ]:
## Make chain and basis
MU,ECT,CAR = kramers_chain(2)
H_Kitaev = MU + ECT + CAR
basis = fst.FockStates(2)
restr_basis = basis.restrict(Ez_inf =False, U_inf=True)

## Set sweet spot
H_Kitaev[CAR[0]] = 20
H_Kitaev[CAR[1]] = -20
H_Kitaev[ECT] = 20*np.sqrt(2)
H_Kitaev[MU] = 0

## Solve
E,phi = np.linalg.eigh(H_Kitaev[restr_basis].to_array(), UPLO='U')
phi = np.transpose(phi)

In [ ]:
fs = fst.FockSystemBase()
N=2
pos_ops = np.arange(4*N)
option_all = generate_combinations(pos_ops)
option_all.append([])

options = generate_combinations_restr(pos_ops)

In [ ]:
phi_even = map_to_new_basis(phi[2], restr_basis, basis)
phi_odd = map_to_new_basis(phi[0], restr_basis,basis)
gamma = constrained_operator(phi_odd, phi_even, options, basis, commutator_oper = n_L)

In [ ]:
gamma

In [ ]:
test_operator(phi[0], phi[2], restr_basis, gamma, verbose=True)

In [ ]:
gamma = min_constrained_operator(phi[0], phi[2], options, restr_basis, commutator_oper = n_L)

In [ ]:
gamma**2

In [ ]:
test_operator(phi[0], phi[2], restr_basis, gamma_constr, verbose=True)

## Three Sites

In [ ]:
c_d_L = fst.OperSequence(0)
c_u_L = fst.OperSequence(2)
c_d_M = fst.OperSequence(4)
c_u_M = fst.OperSequence(6)
c_d_R = fst.OperSequence(8)
c_u_R = fst.OperSequence(10)
n_middle = c_d_M*(~c_d_M) + c_u_M*(~c_u_M)

In [ ]:
fs = fst.FockSystemBase()
N=3
pos_ops = np.arange(4*N)
pos_ops = [0,1,2,3,
           4,5,6,7,
          8,9,10,11]
#pos_ops = [1,2,4,5,10,11]

options = generate_combinations_restr(pos_ops)
#options = gamma.oper_list

In [ ]:
## Make chain and basis
MU,ECT,CAR = kramers_chain(3)
H_Kitaev = MU + ECT + CAR
basis = fst.FockStates(3)
restr_basis = basis.restrict(Ez_inf =False, U_inf=True)
basis_even = restr_basis.restrict('even')
basis_odd = restr_basis.restrict('odd')

phase = np.pi
## Set sweet spot
H_Kitaev[CAR[0]] = 20
H_Kitaev[CAR[1]] = -20
H_Kitaev[CAR[2]] = -20#*np.exp(-1j*phase)
H_Kitaev[CAR[3]] = 20#*np.exp(-1j*phase)

#H_Kitaev[ECT] = 20*np.sqrt(1/(-2*np.cos(phase)))
H_Kitaev[ECT] = 20/np.sqrt(2)
H_Kitaev[MU] = 0

## Solve
E,phi = np.linalg.eigh(H_Kitaev[restr_basis].to_array(),UPLO='U')
phi = np.transpose(phi)
#E_odd,phi_odd = np.linalg.eigh(H_Kitaev[basis_odd].to_array(), UPLO='U')
#phi_odd = np.transpose(phi_odd)

#E_even,phi_even = np.linalg.eigh(H_Kitaev[basis_even].to_array(), UPLO='U')
#phi_even = np.transpose(phi_even)

In [ ]:
new_phi = map_to_new_basis(phi[0], restr_basis, basis)

In [ ]:
phi_even = map_to_new_basis(phi[2], restr_basis, basis)
phi_odd = map_to_new_basis(phi[0], restr_basis,basis)
basic_instance = constrained_operator(phi_odd, phi_even, options, basis, commutator_oper = n_middle)


In [ ]:
short_option = constrained_operator(phi_odd, phi_even, proper_guy, basis, commutator_oper = n_middle)


In [ ]:
test_operator(phi[0], phi[2], restr_basis, short_option, verbose=True)

In [ ]:
is_hermitian(short_option)

In [ ]:
oper_test = iterate_to_optimal_operator(phi_odd,phi_even, options, basis, commutator_oper = n_middle)
print(len(oper_test.oper_list))
oper_test

In [ ]:
oper_test**2

In [ ]:
test_operator(phi_odd, phi_even, basis, oper_test, verbose=True)

In [ ]:
is_hermitian(oper_test)

In [ ]:
commute(oper_test,n_middle)

In [ ]:
operator_1 = min_constrained_operator(phi_odd,phi_even, options, basis, commutator_oper = n_middle)
test_operator(phi_odd, phi_even, basis, operator_1, verbose=True)

In [ ]:
operator_2 = min_constrained_operator(phi_odd, phi_even, options, restr_basis, commutator_oper = n_middle)
test_operator(phi_odd, phi_even, restr_basis, operator_2, verbose=True)

## Super Short

In [ ]:
proper_guy = [[0],
 [1],
 [8],
 [9],
 [5, 4, 2],
 [5, 3, 4],
 [5, 10, 4],
 [11, 5, 4],
 [11, 10, 8],
 [11, 9, 10],
 [5, 3, 4, 2, 0],
 [5, 3, 1, 4, 2],
 [9, 5, 3, 4, 0],
 [5, 1, 8, 4, 2],
 [9, 5, 10, 4, 0],
 [11, 5, 1, 8, 4]]

# Paraferm test

In [ ]:
c_d_L = fst.OperSequence(0)
c_u_L = fst.OperSequence(2)
c_d_R = fst.OperSequence(4)
c_u_R = fst.OperSequence(6)
n_L_u = c_u_L*(~c_u_L)
n_L_d = c_d_L*(~c_d_L)
n_R_u = c_u_R*(~c_u_R)
n_R_d = c_d_R*(~c_d_R)
n_L = c_u_L*(~c_u_L) + c_d_L*(~c_d_L)

a_L_d = (-n_L_u+1)*(~c_d_L)
c_L_d = ~a_L_d
a_L_u = (-n_L_d+1)*(~c_u_L)
c_L_u = ~a_L_u
a_R_d = (-n_R_u+1)*(~c_d_R)
c_R_d = ~a_R_d
a_R_u = (-n_R_d+1)*(~c_u_R)
c_R_u = ~a_R_u

n_R_all = c_R_u*a_R_u + c_R_d*a_R_d
n_L_all = c_L_u*a_L_u + c_L_d*a_L_d
n_L_u_constr =  c_L_u*a_L_u
n_L_d_constr = c_L_d*a_L_d

In [ ]:
paraferm_A = (-n_L_all+1)*(-c_R_d + a_R_u) 
paraferm_B = (c_R_d*a_R_u + 1/np.sqrt(2)*(a_R_d + c_R_u))*c_L_d*a_L_u
paraferm_E = -1/np.sqrt(2)*(n_L_u_constr*c_R_d + n_L_d_constr*a_R_u) 
paraferm_C = -(-(1+np.sqrt(2))/np.sqrt(2)*n_L_all+1)*c_R_u*a_R_d 
paraferm_D = -(-(1+np.sqrt(2))/np.sqrt(2)*n_R_all+1)*c_L_u*a_L_d

In [ ]:
paraferm = paraferm_A+paraferm_B+paraferm_C+paraferm_D+paraferm_E

In [ ]:
test_operator(phi[0], phi[2], restr_basis,paraferm**3)

In [ ]:
state_init = fst.FockStates(states = restr_basis.states, weights = phi[0])


In [ ]:
cubed = paraferm*paraferm*paraferm

In [ ]:
state_A = act_oper_on_state(paraferm,state_init)
state_B = act_oper_on_state(paraferm, state_A)
state_C = act_oper_on_state(paraferm, state_B)
state_C

In [ ]:
paraferm**3

In [ ]:
operator_options = options
basis=restr_basis
phi_A = phi[1]
phi_B = phi[2]

In [ ]:
col_matrix_A = np.round(build_column_matrix(operator_options, phi_A, basis),10)
col_matrix_B = np.round(build_column_matrix(operator_options, phi_B, basis), 10)

M_stacked = np.round(np.vstack([
    col_matrix_A,   # shape (D, M)
    col_matrix_B    # shape (D, M)
],dtype=complex),10)           # shape (2D, M)


In [ ]:
from scipy.linalg import null_space

In [ ]:
nspace = null_space(M_stacked)


In [ ]:
M_stacked

In [ ]:
M_stacked @ np.transpose(nspace)[5]

In [ ]:
nspace